In [ ]:
## Build task_events.tsv for a single task

# Given a `task_label` and the acquisition
# output folder containing `logs.parquet`, emit a BIDS-style `task_events.tsv`
# with columns `onset`, `duration`, `trial_type`. Onsets are seconds relative
# to the first `time_ns` in `logs.parquet` (the recording start).

from pathlib import Path
import pandas as pd

output_folder = Path("/Users/clairenastaskin/data/ARIA/subs/sub-NHS004/0320_task-verbalfluency_run-001/")
ref_table = "task_table.tsv"
logs_path = output_folder / "logs.parquet"
task_table_path = output_folder / ref_table
assert logs_path.exists(), f"logs.parquet not found at {logs_path}"
assert task_table_path.exists(), f"{ref_table} not found at {task_table_path}"
print(f"output_folder   = {output_folder}")

output_folder   = /Users/clairenastaskin/data/ARIA/subs/sub-NHS004/0320_task-verbalfluency_run-001


In [ ]:
### Load recording reference time from `logs.parquet`

# The first `time_ns` in `logs.parquet` is the wall-clock timestamp of the
# first image of the recording; all event onsets are measured relative to it.

logs_df = pd.read_parquet(logs_path, columns=["time_ns"])
recording_start_ns = int(logs_df["time_ns"].min())
tr_seconds = float(logs_df["time_ns"].sort_values().diff().mean()) / 1e9
print(f"recording_start_ns = {recording_start_ns}")
print(f"Tr (avg inter-image interval) = {tr_seconds:.6f} s")

recording_start_ns = 1774001047337428930
Tr (avg inter-image interval) = 2.000013 s


In [ ]:
### Build and write `task_events.tsv`

task_table = pd.read_csv(task_table_path, sep="\t")
required_cols = {"screen_name", "start_time_ns", "duration"}
missing = required_cols - set(task_table.columns)
assert not missing, f"{ref_table} missing required columns: {missing}"

# Set the recording origin to the middle of the first TR.
onset_seconds = (task_table["start_time_ns"].astype("int64") - recording_start_ns) / 1e9 + tr_seconds / 2

events = pd.DataFrame(
    {
        "onset": onset_seconds,
        "duration": task_table["duration"].astype(float),
        "trial_type": task_table["screen_name"],
    }
)

events_path = output_folder / "task_events.tsv"
events.to_csv(events_path, sep="\t", index=False, float_format="%.6f")
print(f"wrote {events_path}")
print(events.to_string(index=False))

wrote /Users/clairenastaskin/data/ARIA/subs/sub-NHS004/0320_task-verbalfluency_run-001/task_events.tsv
     onset  duration  trial_type
 -0.398839 19.769190 instruction
 19.372152 30.271636        wait
 49.657135 29.986963     words_f
 79.645866 29.999614        wait
109.654594 29.991713     words_a
139.648084 29.999638        wait
169.656257 29.991580     words_s
199.649427 29.999476        wait
